In [1]:
import numpy as np
import matplotlib.pyplot as plt
import os
from pathlib import Path
import pandas as pd

import torch
import torch.nn as nn

Set seed and turn of non-deterministic behavior for reproducibility

In [ ]:
SEED = 2026

def set_seed(seed: int = 2026) -> None:
    """Set seed and disable non-deterministic behavior for reproducibility"""
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

Set device

In [67]:
def set_device(device: str = 'cpu') -> str:
    """Set device to CUDA if available. Set to cpu if not."""
    if torch.cuda.is_available():
        device = 'cuda'
    else:
        device = 'cpu'
    return device

device = set_device()

Change working directory

In [2]:
notebook_path = %pwd

os.chdir(Path(notebook_path).parent.parent.parent)
os.getcwd()

'/home/atleeri/repositories/forecast-electricity-markets'

Load Processed Data

In [3]:
processed_data_dir = Path('data/processed/')
filepaths = list(processed_data_dir.glob('**/all_samples/*60*.parquet'))
filepaths

[PosixPath('data/processed/opsd-time_series-2020-10-06/all_samples/time_series_60min_singleindex.parquet')]

In [4]:
filepath = filepaths[0]
df = pd.read_parquet(filepath)
df

,utc_timestamp,cet_cest_timestamp,DE_wind_generation,DE_price_ahead
0,2014-12-31 23:00:00+00:00,2015-01-01 00:00:00+01:00,NaN,NaN
1,2015-01-01 00:00:00+00:00,2015-01-01 01:00:00+01:00,8852.0,NaN
2,2015-01-01 01:00:00+00:00,2015-01-01 02:00:00+01:00,9054.0,NaN
3,2015-01-01 02:00:00+00:00,2015-01-01 03:00:00+01:00,9070.0,NaN
4,2015-01-01 03:00:00+00:00,2015-01-01 04:00:00+01:00,9163.0,NaN
...,...,...,...,...
50396,2020-09-30 19:00:00+00:00,2020-09-30 20:00:00+01:00,10654.0,49.92
50397,2020-09-30 20:00:00+00:00,2020-09-30 21:00:00+01:00,11836.0,42.79
50398,2020-09-30 21:00:00+00:00,2020-09-30 22:00:00+01:00,12168.0,35.02
50399,2020-09-30 22:00:00+00:00,2020-09-30 23:00:00+01:00,12533.0,34.40


## Forecast price ahead using both historical price data and wind generation

In [5]:
valid_mask = df['DE_wind_generation'].notna() & df['DE_price_ahead'].notna()
df_valid = df[valid_mask]
df_valid

,utc_timestamp,cet_cest_timestamp,DE_wind_generation,DE_price_ahead
32856,2018-09-30 23:00:00+00:00,2018-10-01 00:00:00+01:00,6042.0,56.10
32857,2018-10-01 00:00:00+00:00,2018-10-01 01:00:00+01:00,6021.0,51.41
32858,2018-10-01 01:00:00+00:00,2018-10-01 02:00:00+01:00,6342.0,47.38
32859,2018-10-01 02:00:00+00:00,2018-10-01 03:00:00+01:00,7144.0,47.59
32860,2018-10-01 03:00:00+00:00,2018-10-01 04:00:00+01:00,7855.0,51.61
...,...,...,...,...
50395,2020-09-30 18:00:00+00:00,2020-09-30 19:00:00+01:00,8526.0,55.34
50396,2020-09-30 19:00:00+00:00,2020-09-30 20:00:00+01:00,10654.0,49.92
50397,2020-09-30 20:00:00+00:00,2020-09-30 21:00:00+01:00,11836.0,42.79
50398,2020-09-30 21:00:00+00:00,2020-09-30 22:00:00+01:00,12168.0,35.02


Create sequences

In [62]:
def create_sequences(features, targets, input_len: int = 48, horizon: int = 24):
    """Create sequences for seq2seq forecasting
    
    Args:
        features: data to use in forecast
        targets: data to forecast
        input_len: past timesteps to use in forecast (encoder window length)
        horizon: future timesteps to forecast (decoder window length)

    Returns:
        X: (n_samples, input_len, n_features) inputs to encoder
        y: (n_samples, horizon, n_targets) targets (labels) for decoder
    """

    n_samples = len(features)

    X = np.empty((n_samples - input_len - horizon, input_len, features.shape[-1]))
    y = np.empty((n_samples - input_len - horizon, horizon, targets.shape[-1]))

    for i_sample in range(n_samples - input_len - horizon):
        for i_feature in range(features.shape[-1]):
            X[i_sample,:,i_feature] = features[i_sample:(i_sample + input_len), i_feature]

        for i_feature in range(targets.shape[-1]):
            y[i_sample,:,i_feature] = targets[(i_sample + input_len):(i_sample + input_len + horizon), i_feature]

    return np.array(X), np.array(y)

In [78]:
features = df_valid[['DE_wind_generation', 'DE_price_ahead']].values
targets = df_valid['DE_price_ahead'].values

if targets.ndim == 1:
    targets = np.expand_dims(targets, axis=-1)

X, y = create_sequences(features, targets)
X.shape, y.shape

((17468, 48, 2), (17468, 24, 1))

In [79]:
X = torch.tensor(X, dtype = torch.float32)
y = torch.tensor(y, dtype = torch.float32)

In [81]:
X[0,0,0].dtype

torch.float32

Create Seq2Seq Model

In [ ]:
class Seq2SeqGRU(nn.Module):
    def __init__(self, enc_input_size: int = 2, dec_input_size: int = 1, hidden_size: int = 10, num_layers: int = 1,):
        super().__init__()
        self.encoder = nn.GRU(input_size=enc_input_size, hidden_size=hidden_size, batch_first=True, device=device)
        self.decoder = nn.GRU(input_size=dec_input_size, hidden_size=hidden_size, batch_first=True, device=device)
        self.fc = nn.Linear(hidden_size, dec_input_size)

    def forward(self, X, horizon: int = 24):

        # encode input
        _, hidden = self.encoder(X)

        print(hidden.shape)

        # seed for decoder: the last observed price
        dec_input = X[:, -1:, 1:2] # dims: (batch, 1, 1)

        print(X.shape, dec_input.shape)

        predictions = []
        for time_step in range(horizon):
            # decode
            dec_output, hidden = self.decoder(dec_input, hidden)

            # make prediction from output of decoder
            prediction = self.fc(dec_output)
            predictions.append(prediction)

            # am not using teacher forcing for now and just reuse
            dec_input = prediction

        return predictions


In [105]:
model = Seq2SeqGRU(enc_input_size=2)

predictions = model.forward(X)

torch.Size([1, 17468, 10])
torch.Size([17468, 48, 2]) torch.Size([17468, 1, 1])


In [109]:
torch.cat(predictions, dim=1).shape

torch.Size([17468, 24, 1])

In [110]:
len(predictions)

24